In [12]:
# Creamos un archivo CSV con gastos de ejemplo
csv_contenido = """fecha,categoria,descripcion,monto
2024-01-05,Comida,Supermercado,850
2024-01-07,Transporte,Uber,120
2024-01-10,Entretenimiento,Netflix,199
2024-01-12,Comida,Restaurante,350
2024-01-15,Salud,Farmacia,280
2024-01-18,Transporte,Gasolina,600
2024-01-20,Comida,Supermercado,920
2024-01-22,Entretenimiento,Cine,180
2024-01-25,Ropa,Zapatos,1200
2024-01-28,Comida,Restaurante,450
"""

with open('gastos.csv', 'w') as f:
    f.write(csv_contenido)

print("✅ Archivo gastos.csv creado")


✅ Archivo gastos.csv creado


In [13]:
# Leemos el archivo sin usar Pandas (Python puro)
def leer_gastos(archivo):
    gastos = []
    with open(archivo, 'r') as f:
        lineas = f.readlines()
        encabezado = lineas[0].strip().split(',')
        for linea in lineas[1:]:
            valores = linea.strip().split(',')
            gasto = {
                'fecha': valores[0],
                'categoria': valores[1],
                'descripcion': valores[2],
                'monto': float(valores[3])
            }
            gastos.append(gasto)
    return gastos

gastos = leer_gastos('gastos.csv')

print("📋 GASTOS REGISTRADOS:")
print("-" * 45)
for g in gastos:
    print(f"{g['fecha']}  |  {g['categoria']:<15} |  ${g['monto']:.2f}")


📋 GASTOS REGISTRADOS:
---------------------------------------------
2024-01-05  |  Comida          |  $850.00
2024-01-07  |  Transporte      |  $120.00
2024-01-10  |  Entretenimiento |  $199.00
2024-01-12  |  Comida          |  $350.00
2024-01-15  |  Salud           |  $280.00
2024-01-18  |  Transporte      |  $600.00
2024-01-20  |  Comida          |  $920.00
2024-01-22  |  Entretenimiento |  $180.00
2024-01-25  |  Ropa            |  $1200.00
2024-01-28  |  Comida          |  $450.00


In [14]:
def resumen_por_categoria(gastos):
    categorias = {}
    for g in gastos:
        cat = g['categoria']
        if cat not in categorias:
            categorias[cat] = 0
        categorias[cat] += g['monto']
    return categorias

def total_gastos(gastos):
    return sum(g['monto'] for g in gastos)

# Mostramos el resumen
categorias = resumen_por_categoria(gastos)
total = total_gastos(gastos)

print("📊 RESUMEN POR CATEGORÍA:")
print("-" * 35)
for cat, monto in sorted(categorias.items(), key=lambda x: x[1], reverse=True):
    porcentaje = (monto / total) * 100
    print(f"{cat:<20} ${monto:>8.2f}  ({porcentaje:.1f}%)")

print("-" * 35)
print(f"{'TOTAL':<20} ${total:>8.2f}")


📊 RESUMEN POR CATEGORÍA:
-----------------------------------
Comida               $ 2570.00  (49.9%)
Ropa                 $ 1200.00  (23.3%)
Transporte           $  720.00  (14.0%)
Entretenimiento      $  379.00  (7.4%)
Salud                $  280.00  (5.4%)
-----------------------------------
TOTAL                $ 5149.00


In [15]:
def gasto_mas_alto(gastos):
    return max(gastos, key=lambda x: x['monto'])

def agregar_gasto(gastos, fecha, categoria, descripcion, monto):
    nuevo = {
        'fecha': fecha,
        'categoria': categoria,
        'descripcion': descripcion,
        'monto': float(monto)
    }
    gastos.append(nuevo)
    # Guardamos en el CSV
    with open('gastos.csv', 'a') as f:
        f.write(f"\n{fecha},{categoria},{descripcion},{monto}")
    print(f"✅ Gasto agregado: {descripcion} - ${monto}")
    return gastos

# Gasto más alto
mas_alto = gasto_mas_alto(gastos)
print(f"💸 GASTO MÁS ALTO:")
print(f"   {mas_alto['descripcion']} - ${mas_alto['monto']:.2f} ({mas_alto['categoria']})")

# Probamos agregar uno nuevo
gastos = agregar_gasto(gastos, '2024-01-30', 'Comida', 'Pizza', 320)

# Verificamos que se agregó
print(f"\n📋 Total de gastos registrados: {len(gastos)}")


💸 GASTO MÁS ALTO:
   Zapatos - $1200.00 (Ropa)
✅ Gasto agregado: Pizza - $320

📋 Total de gastos registrados: 11


In [17]:
import csv
import os
from datetime import datetime

class Gasto:
    CATEGORIAS_VALIDAS = ['Comida', 'Transporte', 'Salud', 'Entretenimiento', 'Ropa', 'Otro']

    def __init__(self, fecha, categoria, descripcion, monto):
        self.fecha = self._validar_fecha(fecha)
        self.categoria = self._validar_categoria(categoria)
        self.descripcion = descripcion.strip()
        self.monto = self._validar_monto(monto)

    def _validar_fecha(self, fecha):
        try:
            return datetime.strptime(fecha, '%Y-%m-%d')
        except ValueError:
            raise ValueError(f"Fecha inválida: {fecha}. Usa formato YYYY-MM-DD")

    def _validar_categoria(self, categoria):
        if categoria not in self.CATEGORIAS_VALIDAS:
            raise ValueError(f"Categoría inválida. Opciones: {self.CATEGORIAS_VALIDAS}")
        return categoria

    def _validar_monto(self, monto):
        try:
            monto = float(monto)
            if monto <= 0:
                raise ValueError
            return monto
        except ValueError:
            raise ValueError(f"Monto inválido: {monto}. Debe ser número positivo")

    def __str__(self):
        return f"{self.fecha.strftime('%Y-%m-%d')} | {self.categoria:<15} | {self.descripcion:<20} | ${self.monto:>8.2f}"


class AnalizadorGastos:
    def __init__(self, archivo='gastos.csv'):
        self.archivo = archivo
        self.gastos = []
        self._cargar()

    def _cargar(self):
        if not os.path.exists(self.archivo):
            self._crear_csv()
            return
        with open(self.archivo, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                try:
                    self.gastos.append(Gasto(
                        row['fecha'], row['categoria'],
                        row['descripcion'], row['monto']
                    ))
                except ValueError as e:
                    print(f"⚠️  Fila ignorada: {e}")

    def _crear_csv(self):
        with open(self.archivo, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['fecha', 'categoria', 'descripcion', 'monto'])
        print(f"✅ Archivo {self.archivo} creado")

    def agregar(self, fecha, categoria, descripcion, monto):
        gasto = Gasto(fecha, categoria, descripcion, monto)
        self.gastos.append(gasto)
        with open(self.archivo, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                gasto.fecha.strftime('%Y-%m-%d'),
                gasto.categoria,
                gasto.descripcion,
                gasto.monto
            ])
        print(f"✅ Agregado: {gasto}")
        return gasto

    def total(self):
        return sum(g.monto for g in self.gastos)

    def por_categoria(self):
        resumen = {}
        for g in self.gastos:
            resumen[g.categoria] = resumen.get(g.categoria, 0) + g.monto
        return dict(sorted(resumen.items(), key=lambda x: x[1], reverse=True))

    def por_mes(self):
        meses = {}
        for g in self.gastos:
            clave = g.fecha.strftime('%Y-%m')
            meses[clave] = meses.get(clave, 0) + g.monto
        return dict(sorted(meses.items()))

    def gasto_mas_alto(self):
        return max(self.gastos, key=lambda x: x.monto) if self.gastos else None

    def gasto_mas_bajo(self):
        return min(self.gastos, key=lambda x: x.monto) if self.gastos else None

    def promedio(self):
        return self.total() / len(self.gastos) if self.gastos else 0

    def reporte(self):
        if not self.gastos:
            print("No hay gastos registrados.")
            return

        total = self.total()
        separador = "=" * 55

        print(f"\n{separador}")
        print(f"{'📊 REPORTE DE GASTOS':^55}")
        print(separador)

        print(f"\n{'RESUMEN GENERAL':}")
        print(f"  Total gastos:      ${total:>10.2f}")
        print(f"  Promedio por gasto: ${self.promedio():>9.2f}")
        print(f"  Número de gastos:  {len(self.gastos):>10}")

        print(f"\n{'POR CATEGORÍA':}")
        print("-" * 45)
        for cat, monto in self.por_categoria().items():
            barra = '█' * int((monto / total) * 20)
            pct = (monto / total) * 100
            print(f"  {cat:<16} ${monto:>8.2f}  {pct:>5.1f}%  {barra}")

        print(f"\n{'POR MES':}")
        print("-" * 45)
        for mes, monto in self.por_mes().items():
            print(f"  {mes}        ${monto:>10.2f}")

        mas_alto = self.gasto_mas_alto()
        mas_bajo = self.gasto_mas_bajo()
        print(f"\n{'EXTREMOS':}")
        print(f"  💸 Mayor: {mas_alto.descripcion} (${mas_alto.monto:.2f})")
        print(f"  💰 Menor: {mas_bajo.descripcion} (${mas_bajo.monto:.2f})")
        print(f"\n{separador}\n")


print("✅ Clases cargadas correctamente")


✅ Clases cargadas correctamente


In [18]:
# Inicializamos con el CSV que ya tienes
analizador = AnalizadorGastos('gastos.csv')

# Agregamos un gasto nuevo con validación real
analizador.agregar('2024-02-01', 'Salud', 'Gimnasio', 500)

# Generamos el reporte completo
analizador.reporte()


✅ Agregado: 2024-02-01 | Salud           | Gimnasio             | $  500.00

                  📊 REPORTE DE GASTOS                  

RESUMEN GENERAL
  Total gastos:      $   5969.00
  Promedio por gasto: $   497.42
  Número de gastos:          12

POR CATEGORÍA
---------------------------------------------
  Comida           $ 2890.00   48.4%  █████████
  Ropa             $ 1200.00   20.1%  ████
  Salud            $  780.00   13.1%  ██
  Transporte       $  720.00   12.1%  ██
  Entretenimiento  $  379.00    6.3%  █

POR MES
---------------------------------------------
  2024-01        $   5469.00
  2024-02        $    500.00

EXTREMOS
  💸 Mayor: Zapatos ($1200.00)
  💰 Menor: Uber ($120.00)


